In [35]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
import sys
sys.path.append('../../..') #This line makes it possible to access scripts and settings, which are above notebooks in heirarchy
#the above line also makes it possible to access files from the root with out a ROOT_DIR
import subprocess
import importlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from numpy.lib.recfunctions import drop_fields

from notebooks.preprocessing.phase2preprocess import preprocess_general, load_parquet, prpsettings
# from scripts.generate_dataset import generate_dataset
from scripts.dataImport import impsettings

In [36]:
#could we add these constants to a settings .toml file and settings class?
#We could also put all our model kwargs in settings classes and .toml files
BASE_DIR = Path("__file__").resolve().parent
ROOT_DIR = BASE_DIR.parents[1]
ALGORITHM_DIR = ROOT_DIR / "model" / "algorithms"
OUTPUT_DIR = ROOT_DIR / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [37]:
MODEL_FILES = {
    "xgboost": "model_xgboost",
    "mlp": "model_mlp",
    "random_forest": "model_random_forest",
}

In [22]:
def import_module_or_notebook(name):
    py = ALGORITHM_DIR / f"{name}.py"
    nb = ALGORITHM_DIR / f"{name}.ipynb"

    if not py.exists() and not nb.exists():
        raise FileNotFoundError(f"{name}.py or {name}.ipynb not found")

    if nb.exists():
        subprocess.run(
            [sys.executable, "-m", "nbconvert", "--to", "script", "--output", name, str(nb)],
            cwd=ALGORITHM_DIR,
            check=True,
            capture_output=True,
            text=True,
        )

    if str(ALGORITHM_DIR) not in sys.path:
        sys.path.insert(0, str(ALGORITHM_DIR))

    importlib.invalidate_caches()

    if name in sys.modules:
        return sys.modules[name]
    return importlib.import_module(name)

In [23]:
def get_models(names=None):
    if names is None:
        names = list(MODEL_FILES)
    if isinstance(names, str):
        names = [names]
    return {name: import_module_or_notebook(MODEL_FILES[name]) for name in names}

In [24]:
def get_training_data():
    """Get training data for phase 3"""
    #check that phase 2 data has been preprocessed first. Essential to learn preprocessing states.
    input_data = impsettings.PROCESSED_DATA_PATH / "flow_training_sample.parquet"
    prpsettings.load_state()
    if not input_data.is_file():
        raise RuntimeError('Must run generate_dataset() through phase_2 get_training_data() first.')
    if prpsettings.encoder is None or prpsettings.mean is None or prpsettings.std is None:
        raise RuntimeError('Must learn preprocessing states on wwt through phase_2 get_training_data() first.')

    #import phase 3 training data from data/processed
    raw = load_parquet(impsettings.PROCESSED_DATA_PATH, 'flow_training_sample.parquet')

    #seperate out labels
    y = raw['label']
    raw = drop_fields(raw, 'label', asrecarray=True)
    #preprocess training data
    print('preprocessing training data')
    raw_pp = preprocess_general(array=raw, split='Flow')
    #no need to seperate training and testing data. The whole of flow_training_sample is used for training.
    X_train, X_val, y_train, y_val = train_test_split(
        raw_pp, y, test_size=0.15, stratify=y, random_state=42
    )

    return (
        pd.DataFrame(X_train), 
        pd.DataFrame(X_val),
        pd.Series(y_train),
        pd.Series(y_val)
        )

In [25]:
def to_df(X):
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    return pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])

In [26]:
def train_model(name, X_train, X_val, y_train, y_val):
    model_mod = get_models(name)[name]
    
    if name == "xgboost":
        return model_mod.train(
            X_train, y_train,
            X_val, y_val,
            model_kwargs={
                "n_estimators": 300,
                "max_depth": 8,
                "learning_rate": 0.1,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "use_gpu": False,
                "random_state": 42,
            },
            save_path=str(OUTPUT_DIR / "xgboost.joblib"),
        )

    if name == "mlp":
        return model_mod.train(
            X_train, y_train,
            X_val, y_val,
            model_kwargs={
                "hidden_layer_sizes": (128, 64),
                "activation": "relu",
                "alpha": 1e-4,
                "max_iter": 300,
                "early_stopping": True,
                "random_state": 42,
            },
            scale=True,
            save_path=str(OUTPUT_DIR / "mlp.joblib"),
        )

    if name == "random_forest":
        return model_mod.train(
            X_train, y_train,
            X_val, y_val,
            model_kwargs={
                "n_estimators": 200,
                "max_depth": None,
                "min_samples_split": 2,
                "min_samples_leaf": 1,
                "max_features": "sqrt",
                "class_weight": "balanced",
                "random_state": 42,
            },
            save_path=str(OUTPUT_DIR / "random_forest.joblib"),
        )

    raise ValueError(f"Unknown model: {name}")

In [27]:
def train_all(X_train, X_val, y_train, y_val):
    return {name: train_model(name, X_train, X_val, y_train, y_val) for name in MODEL_FILES}

In [28]:
def load_saved_model(name):
    if name == "xgboost":
        return joblib.load(OUTPUT_DIR / "xgboost.joblib")

    if name == "mlp":
        return joblib.load(OUTPUT_DIR / "mlp.joblib")

    if name == "random_forest":
        return joblib.load(OUTPUT_DIR / "random_forest.joblib")

    raise ValueError(f"Unknown model: {name}")

In [29]:
def predict_one(X, name):
    X = to_df(X)
    X_np = X.to_numpy()
    obj = load_saved_model(name)

    if name == "xgboost":
        model = obj["model"]
        le = obj["label_encoder"]
        proba = model.predict_proba(X_np)
        pred_enc = model.predict(X_np)
        pred_lbl = le.inverse_transform(pred_enc)
        score = np.max(proba, axis=1)

    elif name == "mlp":
        model = obj["model"]
        scaler = obj.get("scaler", None)
        le = obj["label_encoder"]
        X_in = scaler.transform(X_np) if scaler is not None else X_np
        proba = model.predict_proba(X_in)
        pred_enc = model.predict(X_in)
        pred_lbl = le.inverse_transform(pred_enc)
        score = np.max(proba, axis=1)

    elif name == "random_forest":
        model = obj["model"]
        le = obj["label_encoder"]
        proba = model.predict_proba(X_np)
        pred_enc = model.predict(X_np)
        pred_lbl = le.inverse_transform(pred_enc)
        score = np.max(proba, axis=1)

    else:
        raise ValueError(f"Unknown model: {name}")

    return {
        "label": pred_lbl,   # decoded strings
        "enc": pred_enc,     # integer codes (optional)
        "score": score,
        "proba": proba,
    }

In [30]:
def majority_vote_labels(label_matrix):
    final_labels = []
    for row in label_matrix:
        counts = Counter(row)
        final_labels.append(counts.most_common(1)[0][0])
    return np.array(final_labels, dtype=object)

In [31]:
def majority_vote_scores(label_matrix, score_matrix, final_labels):
    final_scores = []
    for i in range(len(final_labels)):
        voted_label = final_labels[i]
        row_labels = label_matrix[i]
        row_scores = score_matrix[i]

        matching_scores = [row_scores[j] for j in range(len(row_labels)) if row_labels[j] == voted_label]
        if matching_scores:
            final_scores.append(np.mean(matching_scores))
        else:
            final_scores.append(np.max(row_scores))
    return np.array(final_scores, dtype=float)

In [32]:
def predict_all(X):
    X = to_df(X)

    score_cols = []
    label_cols = []

    for name in MODEL_FILES:
        result = predict_one(X, name)
        score_cols.append(np.asarray(result["score"]))
        label_cols.append(np.asarray(result["label"], dtype=object))

    label_matrix = np.column_stack(label_cols)
    score_matrix = np.column_stack(score_cols)

    final_labels = majority_vote_labels(label_matrix)
    final_scores = majority_vote_scores(label_matrix, score_matrix, final_labels)

    X_array = X.to_numpy()
    scores_array = np.column_stack([score_matrix, final_scores])
    labels_array = np.column_stack([label_matrix, final_labels])

    return X_array, scores_array, labels_array

In [33]:
def predict_phase3(X, model_name=None):
    X = to_df(X)

    if model_name:
        result = predict_one(X, model_name)
        X_array = X.to_numpy()
        scores_array = np.asarray(result["score"]).reshape(-1, 1)
        labels_array = np.asarray(result["label"], dtype=object).reshape(-1, 1)
        return X_array, scores_array, labels_array

    # ensemble path
    X_array, scores_array, labels_array = predict_all(X)
    return X_array, scores_array, labels_array

In [34]:
#X_train_flow, X_val_flow, y_train_flow, y_val_flow = get_training_data()
#trained_models = train_all(X_train_flow, X_val_flow, y_train_flow, y_val_flow)

preprocessing training data
xgboost
<module 'model_xgboost' from '/Users/pauligbinedion/MASTERS/SUMMER/CAPSTONE/ECE597-Capstone-IoT-IDS/notebooks/model/algorithms/model_xgboost.py'>
3.3.0
XGBoost
num_classes: 6
train label range: 0 5
val label range: 0 5

Accuracy: 0.9943 | F1-macro: 0.6742
                 precision    recall  f1-score   support

         Benign       1.00      1.00      1.00     30001
    Brute Force       0.81      0.39      0.52       140
DDoS-HTTP Flood       0.99      0.95      0.97       225
   DNS Spoofing       0.89      0.30      0.44        27
 DoS-HTTP Flood       0.95      0.84      0.89       229
            XSS       1.00      0.12      0.22         8

       accuracy                           0.99     30630
      macro avg       0.94      0.60      0.67     30630
   weighted avg       0.99      0.99      0.99     30630

XGBoost done training...
Saved > /Users/pauligbinedion/MASTERS/SUMMER/CAPSTONE/ECE597-Capstone-IoT-IDS/notebooks/saved_models/xgboost.j

In [ ]:
#This should probably be done on the testing set from phase_2. 
#X_train, X_val, X_test, y_train, y_val, _ = get_training_data()
#X_arr, score_arr, label_arr = predict_phase3(X_test)

In [125]:
#label_arr